# 미분 기초 — 계단·속도계로 이해하는 순간 변화율과 경사하강법

**Zero to 머신러닝 딥러닝 Master — Medium 시리즈 #22** · 원문: 책 01-09 「미분 기초 — 순간 변화율과 경사」

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/otoloism/Zero_to_ML_Master/blob/main/medium-series/22-derivative-basics/22_derivative_basics.ipynb)

계단의 기울기(세로 ÷ 가로)에서 출발해 간격 h를 줄이며 순간 기울기로 수렴하는 과정을 숫자로 보고, 거듭제곱 미분 규칙과 직접 재보기 검산, 기울기를 따라 골짜기 바닥으로 내려가는 경사하강법과 보폭(학습률)의 효과를 코드로 확인합니다.

- 📝 Medium 글: **TODO: Medium 게시 후 URL 입력** (`https://medium.com/p/...`)
- 📘 책 원문: [01-09 미분 기초 — 순간 변화율과 경사](https://wikidocs.net/439748)
- 🏠 [mldict.net](https://mldict.net) · ✍️ [기술 블로그](https://wikidocs.net/blog/@mldict/) · 📚 [책 전체 목차](https://wikidocs.net/book/21464)

## 1. 1단계 — 기울기 = 세로 ÷ 가로

완만한 계단과 가파른 계단의 기울기를 계산해 숫자가 클수록 가파르다는 것을 확인합니다.

In [1]:
# 기울기 = 얼마나 가파른가 = 세로 변화 ÷ 가로 변화
def 기울기(가로, 세로):
    return 세로 / 가로

print("완만한 계단: 가로 4칸, 세로 2칸 →", 기울기(4, 2))
print("가파른 계단: 가로 2칸, 세로 6칸 →", 기울기(2, 6))
print("\n숫자가 클수록 가파르다!")

완만한 계단: 가로 4칸, 세로 2칸 → 0.5
가파른 계단: 가로 2칸, 세로 6칸 → 3.0

숫자가 클수록 가파르다!


## 2. 2단계 — 간격 h를 줄이면 순간 기울기에 수렴

f(x) = x² 의 x = 3 근처 평균 기울기가 h → 0 일 때 6 에 다가가는 것을 표로 봅니다.

In [2]:
# f(x) = x²  (예: 시간 x일 때 이동거리)
def f(x):
    return x ** 2

x = 3            # 이 지점에서의 '순간' 기울기를 알고 싶다

# h(들여다보는 구간)를 점점 줄여 본다
for h in [1, 0.1, 0.01, 0.001]:
    평균기울기 = (f(x + h) - f(x)) / h
    print(f"구간 h={h:<6} → 평균 기울기 = {평균기울기:.4f}")

print("\nh가 0에 가까워질수록 → 6에 딱 붙는다")
print("이 6이 바로 x=3에서의 '순간 기울기(미분값)'")

구간 h=1      → 평균 기울기 = 7.0000
구간 h=0.1    → 평균 기울기 = 6.1000
구간 h=0.01   → 평균 기울기 = 6.0100
구간 h=0.001  → 평균 기울기 = 6.0010

h가 0에 가까워질수록 → 6에 딱 붙는다
이 6이 바로 x=3에서의 '순간 기울기(미분값)'


## 3. 3단계 — 거듭제곱 미분 규칙

(xⁿ)' = n·xⁿ⁻¹ 를 x = 2 에서 n = 2, 3, 4 로 계산합니다.

In [3]:
# 미분 규칙:  (x^n)' = n · x^(n-1)   ("지수를 앞으로 내리고, 지수는 1 줄인다")
def 거듭제곱_미분(n, x):
    return n * x ** (n - 1)

print("(x²)' = 2x   → x=2일 때", 거듭제곱_미분(2, 2))
print("(x³)' = 3x²  → x=2일 때", 거듭제곱_미분(3, 2))
print("(x⁴)' = 4x³  → x=2일 때", 거듭제곱_미분(4, 2))

(x²)' = 2x   → x=2일 때 4
(x³)' = 3x²  → x=2일 때 12
(x⁴)' = 4x³  → x=2일 때 32


## 4. 3단계 — 공식 vs 직접 재보기

공식으로 구한 2x 와 아주 작은 h 로 직접 잰 기울기가 일치하는지 확인합니다.

In [4]:
# 공식으로 구한 미분이 맞는지 '직접 재보기'로 검산
def f(x):
    return x ** 2          # f(x) = x²

def 실제_기울기_재보기(f, x, h=1e-6):
    return (f(x + h) - f(x - h)) / (2 * h)   # 아주 좁은 구간의 기울기

x = 3
print("공식으로:  2x =", 2 * x)
print("직접 재보니:", round(실제_기울기_재보기(f, x), 4))
print("일치하는가?", abs(실제_기울기_재보기(f, x) - 2*x) < 0.001)

공식으로:  2x = 6
직접 재보니: 6.0
일치하는가? True


## 5. 4단계 — 골짜기 바닥 찾아 내려가기

기울기의 반대 방향으로 한 걸음씩 이동해 f(x) = x² 의 바닥(x = 0)에 다가갑니다.

In [5]:
# 골짜기 바닥(가장 낮은 곳) 찾기 = 기울기를 보고 아래로 걷기
def f(x):      return x ** 2      # 골짜기 모양 (바닥은 x=0)
def 기울기(x): return 2 * x       # f의 미분

x = 5.0        # 산 중턱에서 출발
보폭 = 0.2      # 한 번에 얼마나 걸을지 (학습률)

for 걸음 in range(6):
    g = 기울기(x)
    print(f"걸음 {걸음}: 현재 x={x:.3f}, 기울기={g:.3f} → 기울기 반대로 이동")
    x = x - 보폭 * g              # 기울기 '반대' 방향으로 한 발

print(f"\n도착: x={x:.3f} (바닥 x=0에 가까워짐)")

걸음 0: 현재 x=5.000, 기울기=10.000 → 기울기 반대로 이동
걸음 1: 현재 x=3.000, 기울기=6.000 → 기울기 반대로 이동
걸음 2: 현재 x=1.800, 기울기=3.600 → 기울기 반대로 이동
걸음 3: 현재 x=1.080, 기울기=2.160 → 기울기 반대로 이동
걸음 4: 현재 x=0.648, 기울기=1.296 → 기울기 반대로 이동
걸음 5: 현재 x=0.389, 기울기=0.778 → 기울기 반대로 이동

도착: x=0.233 (바닥 x=0에 가까워짐)


## 6. 4단계 — 보폭이 크면? 작으면?

보폭 0.1 과 0.9 로 5걸음씩 걸어 학습률에 따라 도착 위치가 어떻게 달라지는지 비교합니다.

In [6]:
def 기울기(x): return 2 * x

for 보폭 in [0.1, 0.9]:
    x = 5.0
    for _ in range(5):
        x = x - 보폭 * 기울기(x)
    상태 = "천천히 안전하게" if 보폭 < 0.5 else "성큼성큼 빠르게"
    print(f"보폭 {보폭} ({상태}): 5걸음 후 x = {x:.4f}")
print("\n보폭이 너무 크면 골짜기를 건너뛰어 튈 수도 있다(주의)")

보폭 0.1 (천천히 안전하게): 5걸음 후 x = 1.6384
보폭 0.9 (성큼성큼 빠르게): 5걸음 후 x = -1.6384

보폭이 너무 크면 골짜기를 건너뛰어 튈 수도 있다(주의)


## 🔗 함께 보기

- 📘 **책 (WikiDocs)** — [Zero to 머신러닝 딥러닝 Master](https://wikidocs.net/book/21464)
- 🏠 **홈페이지** — [mldict.net](https://mldict.net)
- ✍️ **기술 블로그** — [wikidocs.net/blog/@mldict](https://wikidocs.net/blog/@mldict/)
- 💻 **전체 코드 저장소** — [github.com/otoloism/Zero_to_ML_Master](https://github.com/otoloism/Zero_to_ML_Master)

👉 더 자세한 설명과 그림은 [책 원문](https://wikidocs.net/439748) 에서 확인하세요.